# Laura++ lineshape validation: Flatté, Gounaris–Sakurai, Pole and LASS

This notebook validates four isolated lineshape models:

1. $D^+\to[f_0(980)\to\pi^-\pi^+]\pi^+$ with `Flatte.f0_980()`;
2. $D^+\to[\rho(770)^0\to\pi^-\pi^+]\pi^+$ with `GounarisSakurai()`;
3. $D^+\to[\sigma\to\pi^-\pi^+]\pi^+$ with the Laura++ simple fixed-width pole, `Pole()`;
4. $D^+\to[K_0^*(1430)^0/K\pi\ S\text{-wave}\to K^-\pi^+]\pi^+$ with `LASS()`.

For every case we inspect the pure complex lineshape, the complete symmetrized three-body amplitude on a deterministic `DalitzGrid`, and a 100k-event unweighted toy MC generated from an independent 1M-event phase-space pool.

**Normalization is always deterministic and grid based.** `PhaseSpaceMC` is used only for toy generation.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Flatte, GounarisSakurai, LASS, Pole,
    RealImag, Resonance, ResonanceContext, enable_x64, weighted_resample,
)

enable_x64()

GRID_N = 700
N_POOL = 1_000_000
N_TOY = 100_000

channel_ppp = DecayChannel("D+", ("pi-", "pi+", "pi+"))
channel_kpp = DecayChannel("D+", ("K-", "pi+", "pi+"))

print("pi pi pi masses:", channel_ppp.daughter_masses)
print("K pi pi masses:", channel_kpp.daughter_masses)
print("normalization grid:", GRID_N, "x", GRID_N, "=", GRID_N**2, "points")


In [ ]:
def plot_lineshape(m, z, title, xlabel=r"$m$ [GeV]", vertical_lines=()):
    m, z = np.asarray(m), np.asarray(z)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    axes[0,0].plot(m, z.real); axes[0,0].set_ylabel(r"$\mathrm{Re}\,R(m)$")
    axes[0,1].plot(m, z.imag); axes[0,1].set_ylabel(r"$\mathrm{Im}\,R(m)$")
    axes[1,0].plot(m, np.abs(z)**2); axes[1,0].set_ylabel(r"$|R(m)|^2$")
    axes[1,1].plot(m, np.unwrap(np.angle(z))); axes[1,1].set_ylabel("phase [rad]")
    for ax in axes.flat:
        ax.set_xlabel(xlabel)
        for x, label in vertical_lines:
            ax.axvline(x, ls="--", lw=1, label=label)
    if vertical_lines:
        axes[0,0].legend()
    fig.suptitle(title)
    plt.show()

def model_diagnostics(model, title, projection_label):
    grid = model.normalization_sample
    intensity = np.asarray(model.intensity(grid.as_dict()))
    w = np.asarray(grid.weights) * intensity

    fig, ax = plt.subplots(figsize=(7,6))
    h = ax.hist2d(np.asarray(grid.s12), np.asarray(grid.s13), bins=120, weights=w)
    fig.colorbar(h[3], ax=ax, label=r"grid weight $\times |A|^2$")
    ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]",
           title=title+" — deterministic density")
    plt.show()

    s = np.concatenate([np.asarray(grid.s12), np.asarray(grid.s13)])
    ww = np.concatenate([w, w])
    fig, ax = plt.subplots(figsize=(8,5))
    ax.hist(s, bins=120, weights=ww, histtype="step", lw=1.6)
    ax.set(xlabel=projection_label, ylabel="weighted grid intensity",
           title=title+" — symmetrized projection")
    plt.show()

def make_toy(model, title, projection_label, seed_pool, seed_toy):
    pool = model.generate_phase_space(N_POOL, seed=seed_pool)
    intensity = model.intensity(pool.as_dict())
    target = pool.weights * intensity
    print(title, "finite intensity:", bool(jnp.all(jnp.isfinite(intensity))),
          "finite weights:", bool(jnp.all(jnp.isfinite(target))))
    toy = weighted_resample(jax.random.key(seed_toy), pool, target, N_TOY, replace=True)

    fig, ax = plt.subplots(figsize=(7,6))
    h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
    fig.colorbar(h[3], ax=ax, label="events")
    ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]",
           title=title+f" — {N_TOY:,} toy events")
    plt.show()

    s = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
    fig, ax = plt.subplots(figsize=(8,5))
    ax.hist(s, bins=120, histtype="step", lw=1.6)
    ax.set(xlabel=projection_label, ylabel="entries / bin", title=title+" — toy projection")
    plt.show()
    return toy


## 1. $f_0(980)$ only — Flatté

The coupled $\pi\pi/K\bar K$ width is provided by `Flatte.f0_980()`. The ordinary `width` field is set to zero because it is not used by the Flatté denominator.


In [ ]:
mpi_m, mpi_p, _ = channel_ppp.daughter_masses
flatte = Flatte.f0_980()
f0_context = ResonanceContext(channel_ppp.parent_mass, (mpi_m, mpi_p), mpi_p, 0,
                              0.965, 0.0, 3.0, 3.0)
m = jnp.linspace(mpi_m+mpi_p+1e-5, 1.20, 2500)
z = flatte(m, f0_context)
plot_lineshape(m, z, r"$f_0(980)$ Flatté",
               vertical_lines=((0.965, r"$m_0$"),
                               (2*0.493677, r"$K^+K^-$"),
                               (2*0.497611, r"$K^0\bar K^0$")))

gpi, gk = flatte.widths(m, f0_context)
fig, axes = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
axes[0].plot(np.asarray(m), np.real(np.asarray(gpi)), label=r"Re $\Gamma_{\pi\pi}$")
axes[0].plot(np.asarray(m), np.real(np.asarray(gk)), label=r"Re $\Gamma_{K\bar K}$")
axes[1].plot(np.asarray(m), np.imag(np.asarray(gpi)), label=r"Im $\Gamma_{\pi\pi}$")
axes[1].plot(np.asarray(m), np.imag(np.asarray(gk)), label=r"Im $\Gamma_{K\bar K}$")
for ax in axes: ax.legend(); ax.set_xlabel(r"$m$ [GeV]")
plt.show()

f0_model = DecayModel(channel_ppp, [
    Resonance("f0_980", (0,1), RealImag(1,0), mass=0.965, width=0.0, spin=0,
              lineshape=flatte, resonance_radius=3.0, parent_radius=3.0)
], normalization_resolution=GRID_N, normalization_boundary_resolution=20001)

model_diagnostics(f0_model, r"$D^+\to f_0(980)\pi^+$",
                  r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
toy_f0 = make_toy(f0_model, r"$D^+\to f_0(980)\pi^+$",
                  r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", 3100, 3101)


## 2. $\rho(770)^0$ only — Gounaris–Sakurai


In [ ]:
gs = GounarisSakurai()
rho_context = ResonanceContext(channel_ppp.parent_mass, (mpi_m, mpi_p), mpi_p, 1,
                               0.7693, 0.1502, 3.0, 3.0)
m = jnp.linspace(mpi_m+mpi_p+1e-5, 1.15, 2500)
z = gs(m, rho_context)
plot_lineshape(m, z, r"$\rho(770)^0$ Gounaris--Sakurai",
               vertical_lines=((0.7693, r"$m_0$"),))

rho_model = DecayModel(channel_ppp, [
    Resonance("rho770", (0,1), RealImag(1,0), mass=0.7693, width=0.1502, spin=1,
              lineshape=gs, resonance_radius=3.0, parent_radius=3.0)
], normalization_resolution=GRID_N, normalization_boundary_resolution=20001)

model_diagnostics(rho_model, r"$D^+\to\rho(770)^0\pi^+$",
                  r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
toy_rho = make_toy(rho_model, r"$D^+\to\rho(770)^0\pi^+$",
                   r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", 3200, 3201)


## 3. Broad scalar only — simple Pole

`Pole()` follows the Laura++ simple fixed-width pole (Appendix A, Eq. 37),

\[
R(m)=\frac{1}{m-m_0-i\Gamma_0/2}.
\]

For visualization we use the same broad $\sigma$ mass/width values used in the E791 examples; this is a lineshape test, not a claim that this is the preferred physical model of the $f_0(500)$.


In [ ]:
pole = Pole()
pole_context = ResonanceContext(channel_ppp.parent_mass, (mpi_m, mpi_p), mpi_p, 0,
                                0.478, 0.324, 3.0, 3.0)
m = jnp.linspace(mpi_m+mpi_p+1e-5, 1.20, 2500)
z = pole(m, pole_context)
plot_lineshape(m, z, r"Broad scalar — Laura++ simple Pole",
               vertical_lines=((0.478, r"$m_0$"),))

pole_model = DecayModel(channel_ppp, [
    Resonance("sigma_pole", (0,1), RealImag(1,0), mass=0.478, width=0.324, spin=0,
              lineshape=pole, resonance_radius=3.0, parent_radius=3.0)
], normalization_resolution=GRID_N, normalization_boundary_resolution=20001)

model_diagnostics(pole_model, r"$D^+\to(\pi\pi)_{\rm Pole}\pi^+$",
                  r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
toy_pole = make_toy(pole_model, r"$D^+\to(\pi\pi)_{\rm Pole}\pi^+$",
                    r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", 3300, 3301)


## 4. $K\pi$ S-wave only — LASS

We use $D^+\to K^-\pi^+\pi^+$ so the declared $(K^-\pi^+)$ LASS component is automatically symmetrized over the two identical $\pi^+$.

The LASS amplitude is

\[
R(m)=\frac{m}{q\cot\delta_B-iq}
+e^{2i\delta_B}
\frac{m_0\Gamma_0(m_0/q_0)}
{(m_0^2-m^2)-i m_0\Gamma_0(q/m)(m_0/q_0)},
\]

with $\cot\delta_B=1/(aq)+rq/2$. Here we use $a=2.07~{\rm GeV}^{-1}$, $r=3.32~{\rm GeV}^{-1}$ and explicitly set the slowly-varying term cutoff to $1.7$ GeV for the $D^+$ example.


In [ ]:
mk, mpi, _ = channel_kpp.daughter_masses
lass = LASS(scattering_length=2.07, effective_range=3.32, cutoff=1.7)
lass_context = ResonanceContext(channel_kpp.parent_mass, (mk, mpi), mpi, 0,
                                1.425, 0.270, 3.0, 3.0)
mmax = channel_kpp.parent_mass - mpi
m = jnp.linspace(mk+mpi+1e-5, mmax-1e-5, 3000)
nr, bw = lass.terms(m, lass_context)
full = lass(m, lass_context)

plot_lineshape(m, full, r"$K\pi$ LASS — full amplitude",
               xlabel=r"$m_{K\pi}$ [GeV]",
               vertical_lines=((1.425, r"$m_0$"), (1.7, "NR cutoff")))

fig, axes = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
axes[0].plot(np.asarray(m), np.abs(np.asarray(nr))**2, label="LASS_NR")
axes[0].plot(np.asarray(m), np.abs(np.asarray(bw))**2, label="LASS_BW")
axes[0].plot(np.asarray(m), np.abs(np.asarray(full))**2, label="coherent full")
axes[0].set(xlabel=r"$m_{K\pi}$ [GeV]", ylabel=r"$|R|^2$"); axes[0].legend()
axes[1].plot(np.asarray(m), np.unwrap(np.angle(np.asarray(nr))), label="LASS_NR")
axes[1].plot(np.asarray(m), np.unwrap(np.angle(np.asarray(bw))), label="LASS_BW")
axes[1].plot(np.asarray(m), np.unwrap(np.angle(np.asarray(full))), label="full")
axes[1].set(xlabel=r"$m_{K\pi}$ [GeV]", ylabel="phase [rad]"); axes[1].legend()
plt.show()

lass_model = DecayModel(channel_kpp, [
    Resonance("Kpi_S", (0,1), RealImag(1,0), mass=1.425, width=0.270, spin=0,
              lineshape=lass, resonance_radius=3.0, parent_radius=3.0)
], normalization_resolution=GRID_N, normalization_boundary_resolution=20001)

model_diagnostics(lass_model, r"$D^+\to(K^-\pi^+)_{\rm LASS}\pi^+$",
                  r"$m^2(K^-\pi^+)$ [GeV$^2$]")
toy_lass = make_toy(lass_model, r"$D^+\to(K^-\pi^+)_{\rm LASS}\pi^+$",
                    r"$m^2(K^-\pi^+)$ [GeV$^2$]", 3400, 3401)


## What to inspect

For each model check that:

- the real/imaginary parts and phase evolve smoothly except at physical thresholds/cutoffs;
- the Flatté shows the expected $K\bar K$ cusp;
- the GS phase turns rapidly through the $\rho$ region;
- the simple Pole has the expected fixed-width Lorentzian-like structure in $m$;
- the LASS full amplitude differs from the incoherent sum because the effective-range and resonant terms interfere;
- the deterministic grid density and toy-MC distributions agree visually.
